In [ ]:
import pandas as pd
import os
from model_tuner import loadObjects
from model_metrics import summarize_model_performance

from pathlib import Path

In [ ]:
data_path = "../model_files/"

In [ ]:
# X = pd.read_parquet(os.path.join(data_path, "X.parquet"))
# y = pd.read_parquet(os.path.join(data_path, "y.parquet"))

In [ ]:
from model_metrics.model_registry  import available, load_all, load_model

available()  

In [ ]:
from model_metrics.model_registry import best_per_algo, load_best_per_algo

best_per_algo(metric="valid AUC ROC")
champs = load_best_per_algo(metric="valid AUC ROC")

In [ ]:
champs

In [ ]:
model_rf = champs["rf_income"]
model_dt = champs["dt_income"]
model_lr = champs["lr_income"]

In [ ]:
model_titles = ["Logistic Regression", "Decision Tree", "Random Forest"]
models = [model_lr, model_dt, model_rf]

In [ ]:
thresholds = {"Logistic Regression": next(iter(model_lr.threshold.values())), 
              "Decision Tree": next(iter(model_dt.threshold.values())),
              "Random Forest": next(iter(model_rf.threshold.values()))}

In [ ]:
X_valid = pd.read_parquet(os.path.join(data_path, "X_valid.parquet"))
y_valid = pd.read_parquet(os.path.join(data_path, "y_valid.parquet"))

X_test = pd.read_parquet(os.path.join(data_path, "X_test.parquet"))
y_test = pd.read_parquet(os.path.join(data_path, "y_test.parquet"))

In [ ]:
from model_metrics import summarize_model_performance

model_performance = summarize_model_performance(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    model_type="classification",
    return_df=True,
    model_threshold=thresholds,
)

model_performance

In [ ]:
from model_metrics import show_roc_curve

show_roc_curve(
    model=models,
    X=X_test,
    y=y_test,
    model_title=model_titles,
    decimal_places=2,
    curve_kwgs={
        "CatBoost": {"color": "green", "linewidth": 1},
        "CatBoost_No_Sex": {"color": "orange", "linewidth": 1},
        "XGBoost": {"color": "purple", "linewidth": 1},
        "Random Forest": {"color": "black", "linewidth": 1},
        "Logistic Regression": {"color": "blue", "linewidth": 1},
    },
    linestyle_kwgs={"color": "red", "linestyle": "--"},
    title="ROC Curves: Logistic Regression and Random Forest",
    overlay=True,
)

In [ ]:
from model_metrics import plot_threshold_metrics

plot_threshold_metrics(
    model=model_dt,
    X_test=X_valid,
    y_test=y_valid,
    baseline_thresh=False,
    baseline_kwgs={
        "color": "purple",
        "linestyle": "--",
        "linewidth": 1,
    },
    curve_kwgs={
        "linestyle": "-",
        "linewidth": 1,
    },
    text_wrap=40,
    model_threshold=next(iter(model_dt.threshold.values())),
)